<a href="https://colab.research.google.com/github/carlosferreyra/databricks-bootcamp-2025/blob/main/Class_2_Lab_Notebook_(Argentina_Dataset).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bootcamp Databricks 2025 - Clase 2: Ingesta & Bronze Layer
# MAGIC
# MAGIC **Objetivo:** Ingestar archivos CSV crudos (Raw) hacia nuestra primera tabla Delta (Bronze).
# MAGIC
# MAGIC **Dataset:** Usaremos datos reales del gobierno de Argentina: **Representaciones en el Exterior**.
# MAGIC * **Fuente:** [Datos Argentina - Representaciones](https://datos.gob.ar/dataset/exterior-representaciones-argentinas)
# MAGIC * **URL Directa:** `https://infra.datos.gob.ar/catalog/sspm/dataset/158/distribution/158.1/download/representaciones-argentinas-mundo.csv`
# MAGIC
# MAGIC **Agenda:**
# MAGIC 1. Preparar datos (Descarga automática del CSV).
# MAGIC 2. Ingesta con **PySpark** (Método Ingeniero).
# MAGIC 3. Ingesta con **SQL COPY INTO** (Método Analista).
# MAGIC 4. Explorar **Delta Internals** (Logs y Parquet).

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paso 0: Preparación de Datos (Descarga)
# MAGIC
# MAGIC Vamos a descargar el archivo CSV directamente desde el portal de datos abiertos a nuestro sistema de archivos distribuido (DBFS).
# MAGIC
# MAGIC *Nota: Si la URL falla (común en demos en vivo), usaremos un generador de datos dummy como respaldo.*

# COMMAND ----------

import urllib.request
import os

# 1. Definir URL y Ruta Destino
url = "https://infra.datos.gob.ar/catalog/sspm/dataset/158/distribution/158.1/download/representaciones-argentinas-mundo.csv"
file_path = "/dbfs/FileStore/tables/representaciones_exterior.csv"

# 2. Intentar descargar
try:
    print(f"Descargando datos desde: {url}...")
    urllib.request.urlretrieve(url, file_path)
    print("¡Descarga exitosa!")
except Exception as e:
    print(f"Error descargando: {e}. Generando datos de prueba...")
    # Fallback: Datos simulados si falla internet o la URL
    data = """id,nombre,pais,ciudad,direccion,telefono,email
1,Embajada en España,ESPAÑA,MADRID,Calle de Fernando el Santo 15,+34 91 771 0500,embaesp@mrecic.gov.ar
2,Consulado en Nueva York,ESTADOS UNIDOS,NUEVA YORK,12 West 56th Street,+1 212 603 0400,cnyor@mrecic.gov.ar
3,Embajada en Brasil,BRASIL,BRASILIA,SES Avenida das Nações,+55 61 3212 7600,ebras@mrecic.gov.ar
"""
    with open(file_path, "w") as f:
        f.write(data)

# 3. Verificar
display(dbutils.fs.ls("dbfs:/FileStore/tables/representaciones_exterior.csv"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paso 1: Ingesta con PySpark (Enfoque Ingeniero)
# MAGIC
# MAGIC Leemos el archivo CSV crudo. Noten que los datos reales pueden tener problemas de codificación (tildes, ñ), así que especificamos `encoding`.

# COMMAND ----------

# 1. Definir la ruta fuente (Source)
source_path = "dbfs:/FileStore/tables/representaciones_exterior.csv"

# 2. Leer el CSV
# 'header': true -> usa la primera fila como nombres de columna
# 'inferSchema': true -> intenta adivinar tipos de datos
# 'sep': ',' -> el separador es coma
df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("encoding", "UTF-8") # Importante para datos en español
          .load(source_path))

# Visualizar el DataFrame en memoria
display(df_raw)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Guardar como Tabla Delta
# MAGIC
# MAGIC Persistimos este DataFrame como tabla **Delta**.

# COMMAND ----------

# Nombre de la tabla
table_name = "bronze_representaciones"

# Escribir (Write)
(df_raw.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(table_name))

print(f"Tabla '{table_name}' creada exitosamente.")

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Verificamos con SQL
# MAGIC SELECT * FROM bronze_representaciones LIMIT 10;

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paso 2: Ingesta con SQL (Enfoque Analista)
# MAGIC
# MAGIC Usaremos `COPY INTO`. Primero definimos la tabla vacía con el esquema esperado (basado en lo que vimos en el paso anterior).

# COMMAND ----------

# MAGIC %sql
# MAGIC -- 1. Crear tabla vacía (Schema on Write)
# MAGIC CREATE TABLE IF NOT EXISTS bronze_representaciones_sql
# MAGIC (
# MAGIC   id STRING,
# MAGIC   nombre STRING,
# MAGIC   pais STRING,
# MAGIC   ciudad STRING,
# MAGIC   direccion STRING,
# MAGIC   telefono STRING,
# MAGIC   email STRING,
# MAGIC   sitio_web STRING -- Agregamos columnas extra que suele traer el dataset
# MAGIC )
# MAGIC USING DELTA;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- 2. Cargar datos
# MAGIC COPY INTO bronze_representaciones_sql
# MAGIC FROM 'dbfs:/FileStore/tables/representaciones_exterior.csv'
# MAGIC FILEFORMAT = CSV
# MAGIC FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
# MAGIC COPY_OPTIONS ('mergeSchema' = 'true'); -- Esto permite que si hay columnas nuevas, las agregue

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT pais, ciudad, nombre FROM bronze_representaciones_sql WHERE pais = 'ESPAÑA';

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paso 3: Delta Lake Internals
# MAGIC
# MAGIC Vamos a ver la estructura física de la tabla `bronze_representaciones`.

# COMMAND ----------

# MAGIC %sql
# MAGIC DESCRIBE DETAIL bronze_representaciones;

# COMMAND ----------

# Obtenemos la ubicación física
location = spark.sql("DESCRIBE DETAIL bronze_representaciones").first().location
print(f"Archivos guardados en: {location}")

# Listamos la carpeta para ver los Parquet y el _delta_log
display(dbutils.fs.ls(location))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paso 4: History & Time Travel
# MAGIC
# MAGIC Vamos a simular un "error" operativo: Borrar todas las embajadas en BRASIL.

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT count(*) FROM bronze_representaciones WHERE pais = 'BRASIL';

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Ups, borramos datos
# MAGIC DELETE FROM bronze_representaciones WHERE pais = 'BRASIL';

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Verificamos que ya no están
# MAGIC SELECT count(*) FROM bronze_representaciones WHERE pais = 'BRASIL';

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Vamos al historial para recuperar la versión anterior
# MAGIC DESCRIBE HISTORY bronze_representaciones;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Viajamos en el tiempo a la Versión 0 (o la que corresponda antes del DELETE)
# MAGIC SELECT * FROM bronze_representaciones VERSION AS OF 0 WHERE pais = 'BRASIL';

# COMMAND ----------

# MAGIC %md
# MAGIC ## Restaurar Datos (Restore)
# MAGIC
# MAGIC Si el error fue grave, podemos restaurar la tabla completa a su estado original.

# COMMAND ----------

# MAGIC %sql
# MAGIC RESTORE TABLE bronze_representaciones TO VERSION AS OF 0;

# COMMAND ----------

# MAGIC %md
# MAGIC ## Conclusión Lab 1
# MAGIC
# MAGIC 1. Descargamos datos reales de **Argentina**.
# MAGIC 2. Creamos una tabla **Bronze** inmutable (idealmente) usando PySpark y SQL.
# MAGIC 3. Vimos cómo Delta maneja las transacciones y nos salvó de un `DELETE` accidental.